In [1]:
"""
MSTD symbolic analyzer.

Searches for More-Sums-Than-Differences (MSTD) sets by branching over symbolic
collisions and tracking phi, the largest prime characteristic in which a redundant
collision can occur (0=0).  See the accompanying paper for the underlying theory.

An expression (Expr) is a sorted tuple of (variable, coefficient) pairs. For instance 2a+b is an expression, and would be input here as '2*a+b'.
A configuration (Config) is a sorted tuple of Exprs. For instance, it could be (2a+b, b-a), which is input as ['2*a+b', 'b-a']
"""
from __future__ import annotations

import math
import time
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple

_UNSET = object()

# ----------------------------------------------------------------------------
# Type aliases
# ----------------------------------------------------------------------------
Expr = Tuple[Tuple[str, int], ...]   # sorted tuple of (var, coeff) pairs
Config = Tuple[Expr, ...]             # canonicalized: tuple of Exprs, sorted


# ----------------------------------------------------------------------------
# Expression helpers
# ----------------------------------------------------------------------------
def expr_from_dict(d: Dict[str, int]) -> Expr:
    """This function builds a canonical expression from a {variable: coefficient}
    dictionary, dropping zero terms and sorting by variable name."""
    return tuple(sorted((v, c) for v, c in d.items() if c != 0))


def expr_add(a: Expr, b: Expr) -> Expr:
    """This function adds two expressions together, summing the coefficients of
    matching variables and returning the result in canonical form."""
    i = 0
    j = 0
    la = len(a)
    lb = len(b)
    out: List[Tuple[str, int]] = []
    while i < la and j < lb:
        va, ca = a[i]
        vb, cb = b[j]
        if va < vb:
            out.append(a[i])
            i += 1
        elif vb < va:
            out.append(b[j])
            j += 1
        else:
            c = ca + cb
            if c != 0:
                out.append((va, c))
            i += 1
            j += 1
    if i < la:
        out.extend(a[i:])
    if j < lb:
        out.extend(b[j:])
    return tuple(out)


def expr_sub(a: Expr, b: Expr) -> Expr:
    """This function subtracts one expression from another, subtracting the
    coefficients of matching variables and returning the result in canonical
    form."""
    i = 0
    j = 0
    la = len(a)
    lb = len(b)
    out: List[Tuple[str, int]] = []
    while i < la and j < lb:
        va, ca = a[i]
        vb, cb = b[j]
        if va < vb:
            out.append(a[i])
            i += 1
        elif vb < va:
            out.append((vb, -cb))
            j += 1
        else:
            c = ca - cb
            if c != 0:
                out.append((va, c))
            i += 1
            j += 1
    if i < la:
        out.extend(a[i:])
    while j < lb:
        vb, cb = b[j]
        out.append((vb, -cb))
        j += 1
    return tuple(out)


def expr_neg(a: Expr) -> Expr:
    """This function negates an expression by flipping the sign of every
    coefficient, used when considering a difference in the opposite direction."""
    return tuple((v, -c) for v, c in a)


def expr_scale(a: Expr, k: int) -> Expr:
    """This function multiplies every coefficient of an expression by an integer
    k, used to dilate a set or clear a denominator before substituting a
    variable."""
    if k == 1:
        return a
    if k == 0:
        return ()
    return tuple((v, c * k) for v, c in a)


def expr_gcd_reduce(a: Expr) -> Tuple[Expr, int]:
    """This function divides an expression by the GCD of its coefficients,
    returning the reduced expression and that divisor.  The divisor reveals when a
    raw collision is a smaller relation scaled up (e.g. 3a-6b is 3 times a-2b),
    which is what matters for the phi certificate."""
    if not a:
        return a, 1
    # Fast path: if any coefficient is +-1, the GCD must be 1
    for _, c in a:
        if c == 1 or c == -1:
            return a, 1
    if len(a) == 1:
        # Single term: GCD is |coeff|. Reducing gives coefficient sign(c) = +-1.
        v, c = a[0]
        g = abs(c)
        if g == 1:
            return a, 1
        return ((v, 1 if c > 0 else -1),), g
    g = abs(a[0][1])
    for _, c in a[1:]:
        g = math.gcd(g, abs(c))
        if g == 1:
            return a, 1
    if g <= 1:
        return a, 1
    return tuple((v, c // g) for v, c in a), g


def expr_canonical_signed(a: Expr) -> Expr:
    """This function fixes an expression's overall sign by negating it when its
    leading coefficient is negative, so an equation and its negation share one
    collision key."""
    if a and a[0][1] < 0:
        return expr_neg(a)
    return a


def expr_to_str(a: Expr) -> str:
    """This function turns an expression into a readable string such as '2*a-3*c'
    for status output and logs."""
    if not a:
        return "0"
    parts: List[str] = []
    for v, c in a:
        if c == 1:
            parts.append(f"+{v}")
        elif c == -1:
            parts.append(f"-{v}")
        elif c > 0:
            parts.append(f"+{c}*{v}")
        else:
            parts.append(f"{c}*{v}")  # negative includes its sign
    out = "".join(parts)
    if out.startswith("+"):
        out = out[1:]
    return out


# ----------------------------------------------------------------------------
# String parsing (only for user input)
# ----------------------------------------------------------------------------
def parse_expr_from_str(s: str) -> Expr:
    """This function parses a human-typed string such as '2*a - 3*b + e' into the
    internal expression form; bare constant terms are ignored."""
    s = s.strip().replace(' ', '')
    if s == '' or s == '0':
        return ()
    if s[0] not in '+-':
        s = '+' + s
    d: Dict[str, int] = {}
    i = 0
    n = len(s)
    while i < n:
        # Sign
        sign = 1
        while i < n and s[i] in '+-':
            if s[i] == '-':
                sign = -sign
            i += 1
        # Optional integer coefficient
        coeff_str = ''
        while i < n and s[i].isdigit():
            coeff_str += s[i]
            i += 1
        # Optional '*'
        if i < n and s[i] == '*':
            i += 1
        # Variable name (alphanumeric, starting alpha or underscore)
        var_str = ''
        while i < n and (s[i].isalpha() or s[i] == '_' or (var_str and s[i].isdigit())):
            var_str += s[i]
            i += 1
        if not var_str and not coeff_str:
            if i < n:
                i += 1
            continue
        coeff = (int(coeff_str) if coeff_str else 1) * sign
        if var_str:
            d[var_str] = d.get(var_str, 0) + coeff
        # Constants (no var) are ignored as 0
    return expr_from_dict(d)


# ----------------------------------------------------------------------------
# Configuration layer
# ----------------------------------------------------------------------------
def config_canonical(cfg: Config) -> Config:
    """This function sorts a configuration's elements into canonical order so that
    two configurations listing the same elements are recognized as identical
    (variable renaming is handled separately by config_dedup_key). Note that dedup stands
    for de-duplication"""
    return tuple(sorted(cfg))


def config_dedup_key(cfg: Config) -> Tuple[Tuple[int, ...], ...]:
    """This function computes a dedup key for a configuration that is invariant
    under renaming and reordering its variables, so configurations that differ
    only by which variable is called what collapse to the same key."""
    vars_used = sorted(config_vars(cfg))
    k = len(vars_used)
    # Build the column-vectors representation: each element -> tuple of
    # length k where position i is the coefficient of vars_used[i].
    rows: List[Tuple[int, ...]] = []
    for e in cfg:
        d = dict(e)
        rows.append(tuple(d.get(v, 0) for v in vars_used))
    # k = 0 (no variables, all elements are 0): only one row (0,), trivially
    # canonical (and any such cfg has duplicates anyway).
    if k == 0:
        # Just return a key indicating "all zeros, length-1 entries"
        return (tuple(rows),)
    if k == 1:
        # Single-variable leaf: also canonicalize by translation, so that
        # the minimum coefficient becomes 0. This makes configurations that
        # differ only by an additive constant (i.e., translations of the
        # same integer set) dedup to the same key. Translation preserves
        # |A+A| and |A-A| so this doesn't lose MSTD-ness.
        # rows are tuples of length 1; flatten to scalars, shift, re-wrap.
        flat = [r[0] for r in rows]
        m = min(flat)
        return tuple(sorted((c - m,) for c in flat))
    # For k >= 2, try permutations of variable positions to find the
    # lex-smallest representation.
    #
    # IMPORTANT OPTIMIZATION: instead of trying all k! permutations, compute
    # a "signature" per variable (the sorted tuple of its coefficients across
    # all elements) and only permute among variables with IDENTICAL signatures.
    # Variables with distinct signatures can't be swapped without changing the
    # config — so the search space collapses dramatically in the common case
    # where all variables have distinct signatures (single canonical form).
    from itertools import permutations as _perms
    # Per-variable signature: sorted tuple of coefficients across all rows
    # for that column.
    sigs: List[Tuple[int, ...]] = []
    for col_idx in range(k):
        col = sorted(r[col_idx] for r in rows)
        sigs.append(tuple(col))
    # Group column indices by signature
    sig_groups: Dict[Tuple[int, ...], List[int]] = {}
    for idx, sig in enumerate(sigs):
        sig_groups.setdefault(sig, []).append(idx)
    # The canonical variable ordering puts groups in lex-ascending sig order.
    # Within each group, the columns can be in any order (we'll permute them
    # to find lex-smallest).
    sorted_sigs = sorted(sig_groups.keys())
    fixed_order: List[List[int]] = [sig_groups[s] for s in sorted_sigs]
    # Now generate all canonical permutations by permuting WITHIN each group.
    # Cartesian product of per-group permutations.

    def _all_orderings(groups: List[List[int]]):
        if not groups:
            yield ()
            return
        head, *rest = groups
        for head_perm in _perms(head):
            for tail in _all_orderings(rest):
                yield head_perm + tail

    best: Optional[Tuple[Tuple[int, ...], ...]] = None
    for perm in _all_orderings(fixed_order):
        permuted_rows = tuple(tuple(r[i] for i in perm) for r in rows)
        candidate = tuple(sorted(permuted_rows))
        if best is None or candidate < best:
            best = candidate
    return best  # type: ignore


def config_has_duplicates(cfg: Config) -> bool:
    """This function checks whether two elements of a configuration are identical,
    which marks an invalid set and lets the search abandon that branch."""
    return len(set(cfg)) != len(cfg)


def config_vars(cfg: Config) -> Set[str]:
    """This function returns the set of variables still appearing in a
    configuration; its size (the number of free variables left) determines when a
    configuration is a leaf."""
    out: Set[str] = set()
    for e in cfg:
        for v, _ in e:
            out.add(v)
    return out


def config_scale(cfg: Config, k: int) -> Config:
    """This function multiplies every element of a configuration by an integer k,
    used to clear a denominator before eliminating a variable."""
    if k == 1:
        return cfg
    return tuple(expr_scale(e, k) for e in cfg)


def config_gcd_reduce(cfg: Config) -> Tuple[Config, int]:
    """This function divides a whole configuration by the GCD of all its
    coefficients, so sets differing only by a uniform scaling are recognized as
    the same.  This divisor is deliberately excluded from the phi certificate,
    since uniform scaling never changes whether a set is MSTD."""
    g = 0
    for e in cfg:
        for _, c in e:
            g = math.gcd(g, abs(c))
            if g == 1:
                return cfg, 1
    if g <= 1:
        return cfg, 1
    new_cfg = tuple(tuple((v, c // g) for v, c in e) for e in cfg)
    return new_cfg, g


def config_to_str(cfg: Config) -> str:
    """This function renders a whole configuration as a readable string such as
    '[0, a, b-c, 2*a+e]', which the program uses in its status output and logs."""
    return '[' + ', '.join(expr_to_str(e) for e in cfg) + ']'


# ----------------------------------------------------------------------------
# Collision key generation
# ----------------------------------------------------------------------------
def expr_mod_p(a: Expr, p: int) -> Expr:
    """This function reduces every coefficient of an expression mod p into
    {0,...,p-1}, dropping any that become 0."""
    return tuple(sorted((v, c % p) for v, c in a if c % p != 0))


def expr_canonical_modp(a: Expr, p: int) -> Expr:
    """This function returns a canonical mod-p form of an expression by reducing
    mod p and scaling its leading coefficient to 1, so expressions defining the
    same relation mod p (e.g. f+g and 6f+6g mod 7) dedup to one form."""
    am = expr_mod_p(a, p)
    if not am:
        return ()
    lead = am[0][1]
    linv = pow(lead, p - 2, p)   # modular inverse, p prime
    return tuple((v, (c * linv) % p) for v, c in am)


def is_collision_valid_modp(k: Expr, p: int) -> bool:
    """This function decides whether a canonical mod-p key is a usable collision:
    it needs at least 2 variables and, for a 2-variable key, must not force two
    distinct variables equal (coefficients summing to 0 mod p)."""
    if len(k) < 2:
        return False
    if len(k) == 2 and (k[0][1] + k[1][1]) % p == 0:
        return False
    return True


def is_collision_valid_expr(k: Expr) -> bool:
    """This function decides whether a collision key is usable: it needs at least
    2 variables and, for a 2-variable key, coefficients that are not
    equal-and-opposite (which would force two variables to be equal)."""
    if len(k) < 2:
        return False
    if len(k) == 2 and k[0][1] == -k[1][1]:
        return False
    return True


def get_collision_keys(
    cfg: Config, mod_p: Optional[int] = None
) -> Tuple[Dict[Expr, int], Set[int]]:
    """This function returns all usable collision keys for a configuration
    together with reduction_factors, the set of GCDs pulled from EVERY raw
    collision before the validity check (so a spurious coincidence like 3a-3b in
    characteristic 3 is still recorded).  The reduction_factors set is what gates
    phi; the per-key GCDs are branching bookkeeping only."""
    # Unordered pairwise differences (i < j); both i-j and j-i are negatives,
    # so we only need one direction since we'll sign-normalize.
    diffs: List[Expr] = []
    seen: Set[Expr] = set()
    n = len(cfg)
    for i in range(n):
        for j in range(i + 1, n):
            d = expr_sub(cfg[i], cfg[j])
            if d and d not in seen:
                seen.add(d)
                diffs.append(d)
    if mod_p is not None:
        # Store the difference set mod p (coeffs in {0,...,p-1}); differences
        # equal mod p merge, so no "redundant" collision is ever formed between
        # them.  Keys are canonicalized mod p and are always primitive (gcd 1).
        modp_diffs = set()
        for d in diffs:
            dm = expr_mod_p(d, mod_p)
            if dm:
                modp_diffs.add(dm)
        mdiffs = list(modp_diffs)
        keys: Dict[Expr, int] = {}
        m = len(mdiffs)
        for i in range(m):
            d1 = mdiffs[i]
            for j in range(m):
                if i == j:
                    continue
                d2 = mdiffs[j]
                for raw in (expr_sub(d1, d2), expr_add(d1, d2)):
                    ck = expr_canonical_modp(raw, mod_p)
                    if is_collision_valid_modp(ck, mod_p):
                        keys[ck] = 1
        # Mod-p mode: single-variable coincidences carry no reduction to track
        # (differences already live in {0,...,p-1}).  Return an empty set.
        return keys, set()
    keys: Dict[Expr, int] = {}
    reduction_factors: Set[int] = set()
    n_diffs = len(diffs)
    for i in range(n_diffs):
        d1 = diffs[i]
        for j in range(n_diffs):
            if i == j:
                continue
            d2 = diffs[j]
            # d1 = d2  =>  raw = d1 - d2 ;  d1 = -d2  =>  raw = d1 + d2
            for raw in (expr_sub(d1, d2), expr_add(d1, d2)):
                if not raw:
                    continue
                # Reduction factor of THIS raw collision -- computed and recorded
                # ALWAYS, BEFORE any validity check.  A raw like 3a-3b reduces to
                # the "invalid" a=b, but 3(a-b)=0 still holds in characteristic 3,
                # so 3 must gate the certificate.  Likewise 7b=0 -> 7 and
                # 17a-17c -> 17.  This subsumes the single-variable k*a=0 case
                # (gcd of a single term is |k|).
                kr, g = expr_gcd_reduce(raw)
                if g > 1:
                    reduction_factors.add(g)
                # Only VALID collisions (>= 2 vars, not equal-and-opposite)
                # become usable branching keys.
                if is_collision_valid_expr(raw):
                    ck = expr_canonical_signed(kr)
                    prev = keys.get(ck)
                    if prev is None or g < prev:
                        keys[ck] = g
    return keys, reduction_factors


# ----------------------------------------------------------------------------
# Applying a collision
# ----------------------------------------------------------------------------
def plan_elimination(key: Expr, cfg: Config) -> Tuple[str, int, Dict[str, int]]:
    """This function chooses which variable a collision should eliminate and
    returns (elim_var, scale, sub), meaning that after scaling the config by
    scale, scale*elim_var is replaced by the substitution sub."""
    # Count appearances in cfg
    counts: Dict[str, int] = defaultdict(int)
    for e in cfg:
        for v, _ in e:
            counts[v] += 1
    # Smallest |coeff|, tiebreak by least frequent in cfg
    min_abs = min(abs(c) for _, c in key)
    candidates = [(v, c) for v, c in key if abs(c) == min_abs]
    candidates.sort(key=lambda vc: counts.get(vc[0], 0))
    elim_var, elim_coeff = candidates[0]
    scale = abs(elim_coeff)
    sign = 1 if elim_coeff > 0 else -1
    # From elim_coeff * elim_var + sum c_v * v = 0:
    #   |elim_coeff| * sign * elim_var = -sum c_v * v
    #   scale * elim_var = -sign * sum c_v * v
    sub: Dict[str, int] = {}
    for v, c in key:
        if v == elim_var:
            continue
        sub[v] = -sign * c
    return elim_var, scale, sub


def apply_collision(cfg: Config, key: Expr, mod_p: Optional[int] = None) -> Tuple[Optional[Config], int]:
    """This function applies a collision key by eliminating its variable and
    returns (new_cfg, step_reduction), where step_reduction is the GCD divided out
    during this step (or (None, 1) on internal failure)."""
    # Reduce the key by its GCD first
    key_red, key_gcd = expr_gcd_reduce(key)
    elim_var, scale, sub = plan_elimination(key_red, cfg)
    scaled = config_scale(cfg, scale)
    sub_expr = expr_from_dict(sub)
    new_cfg_list: List[Expr] = []
    for e in scaled:
        d: Dict[str, int] = dict(e)
        if elim_var not in d:
            new_cfg_list.append(e)
            continue
        c = d.pop(elim_var)
        # c should be divisible by scale (since we scaled the whole config)
        if c % scale != 0:
            return None, 1
        mult = c // scale
        for v, sc in sub_expr:
            d[v] = d.get(v, 0) + mult * sc
        new_cfg_list.append(expr_from_dict(d))
    new_cfg = tuple(new_cfg_list)
    if mod_p is not None:
        # Mod-p mode: keys are primitive (leading coeff 1) so scale == 1 and no
        # config-scale gcd arises; just reduce coefficients into {0,...,p-1}.
        reduced_cfg = tuple(expr_mod_p(e, mod_p) for e in new_cfg)
        return reduced_cfg, key_gcd
    reduced_cfg, cfg_gcd = config_gcd_reduce(new_cfg)
    step_reduction = max(key_gcd, cfg_gcd)
    return reduced_cfg, step_reduction


# ----------------------------------------------------------------------------
# Interchangeability (top-level only)
#
# Symmetry groups are supplied explicitly by the caller (interchangeable_groups)
# and validated against the initial config; occurrence count is NOT used as an
# automatic symmetry test.
# ----------------------------------------------------------------------------
def collapse_key_for_interchangeability(k: Expr, groups: List[List[str]]) -> Expr:
    """This function returns a canonical form for a collision key under
    permutations within each interchangeable group (and a global sign flip), so
    that keys related by those symmetries collapse to the same form."""
    if not groups:
        return expr_canonical_signed(k)
    group_sets = [set(g) for g in groups]
    all_inter: Set[str] = set().union(*group_sets)
    fixed: Dict[str, int] = {v: c for v, c in k if v not in all_inter}

    def build(sign: int) -> Expr:
        out: Dict[str, int] = {v: sign * c for v, c in fixed.items()}
        for g, gs in zip(groups, group_sets):
            coeffs = sorted((sign * c for v, c in k if v in gs), reverse=True)
            for name, coeff in zip(sorted(g), coeffs):
                out[name] = coeff
        return expr_from_dict(out)

    a = build(1)
    b = build(-1)   # global sign flip
    return a if a <= b else b


def filter_interchangeable(keys: List[Expr], groups: List[List[str]]) -> List[Expr]:
    """Pick one representative per equivalence class under within-group
    permutations (and global sign flip) for the given interchangeable `groups`.

    If `groups` is empty/None, NO collapsing is performed (every key is kept) --
    this is the safe default, since occurrence count alone is not a symmetry
    proof.  Supply groups only for variables whose symmetry is justified by the
    configuration (validated in Analyzer._validate_interchangeable_groups).

    Representative choice: among equivalent keys, prefer the one whose
    interchangeable-variable name-set is lex-first, then the lex-smallest Expr.
    """
    if not groups:
        return list(set(keys))
    inter_set: Set[str] = set().union(*[set(g) for g in groups])

    def representative_priority(k: Expr) -> Tuple:
        used_vars = tuple(sorted(v for v, c in k if v in inter_set))
        return (used_vars, k)

    classes: Dict[Expr, List[Expr]] = {}
    for k in keys:
        c = collapse_key_for_interchangeability(k, groups)
        if not c:
            continue
        classes.setdefault(c, []).append(k)
    out: List[Expr] = []
    for class_members in classes.values():
        out.append(min(class_members, key=representative_priority))
    return out


# ----------------------------------------------------------------------------
# Leaf diameter and reduction-factor utilities
# ----------------------------------------------------------------------------
def distinct_prime_factors(n: int) -> Set[int]:
    """This function returns the set of distinct prime factors of |n| (empty when
    |n| is 0 or 1).  The program uses it to spread a single reduction factor
    across all of the prime characteristics that it affects."""
    n = abs(n)
    out: Set[int] = set()
    while n % 2 == 0:
        out.add(2)
        n //= 2
    i = 3
    while i * i <= n:
        while n % i == 0:
            out.add(i)
            n //= i
        i += 2
    if n > 2:
        out.add(n)
    return out


def largest_prime_factor(n: int) -> int:
    """This function returns the largest prime factor of |n| (and 1 when |n| is 0
    or 1).  It is how the program converts a raw reduction factor into the single
    prime it contributes to the phi certificate."""
    if n <= 1:
        return 1
    mp = 1
    while n % 2 == 0:
        mp = 2
        n //= 2
    i = 3
    while i * i <= n:
        while n % i == 0:
            mp = i
            n //= i
        i += 2
    if n > 2:
        mp = n
    return mp


def _leaf_coeffs(cfg: Config) -> Optional[List[int]]:
    """This function pulls out the plain list of integer coefficients from a leaf
    configuration -- one whose elements all involve at most a single variable --
    which is the concrete integer set the MSTD tests then examine.  It returns
    None when the configuration is not a valid leaf."""
    reduced, _ = config_gcd_reduce(cfg)
    coeffs: List[int] = []
    for e in reduced:
        if not e:
            coeffs.append(0)
        elif len(e) == 1:
            coeffs.append(e[0][1])
        else:
            return None
    return coeffs if coeffs else None


# Prime sieve, lazily extended.
_PRIME_LIST: List[int] = []
_PRIME_LIST_LIMIT: int = 0


def _primes_up_to(limit: int) -> List[int]:
    """This function returns every odd prime below the given limit, starting at 3,
    which supplies the list of characteristics the leaf MSTD test needs to check.
    Results are cached (memoized) so repeated calls stay cheap."""
    global _PRIME_LIST, _PRIME_LIST_LIMIT
    if limit > _PRIME_LIST_LIMIT:
        new_limit = max(limit * 2, 1024)
        sieve = bytearray([1]) * (new_limit + 1)
        sieve[0] = sieve[1] = 0
        for i in range(2, int(new_limit**0.5) + 1):
            if sieve[i]:
                for j in range(i * i, new_limit + 1, i):
                    sieve[j] = 0
        _PRIME_LIST = [p for p in range(3, new_limit + 1) if sieve[p]]
        _PRIME_LIST_LIMIT = new_limit
    # Binary-search index of largest prime < limit
    lo, hi = 0, len(_PRIME_LIST)
    while lo < hi:
        mid = (lo + hi) // 2
        if _PRIME_LIST[mid] < limit:
            lo = mid + 1
        else:
            hi = mid
    return _PRIME_LIST[:lo]


def check_mstd_primes(cfg: Config) -> List[int]:
    """Check MSTD-in-Z/pZ for a leaf config across all relevant primes.

    For each odd prime p <= 2*diam (= 2M - 2m), where diam = max - min of the
    coefficients, evaluate the coefficients mod p.  If they remain distinct mod
    p (so this is a genuine |A|-element set in Z/pZ) and |A+A| > |A-A|, record p.

    Implementation uses bitmask sumsets / diffsets: for prime p, the
    sumset and diffset each fit in a Python int treated as a bitmask of
    length p.  Bitwise-OR replaces set.add, and popcount via bin().count
    replaces len.  Much faster than the set-based version for moderate p.

    Returns the (possibly empty) list of primes in which this leaf is MSTD.
    Returns empty list if cfg isn't a valid leaf.
    """
    coeffs = _leaf_coeffs(cfg)
    if coeffs is None:
        return []
    n = len(coeffs)
    diam = max(coeffs) - min(coeffs)
    # Manuscript range is |A| < p <= 2M - 2m = 2*diam.  _primes_up_to returns
    # primes STRICTLY below its limit, so use 2*diam + 1 to INCLUDE the endpoint
    # p = 2*diam on the rare occasion it is prime (only possible when diam == 1,
    # since 2*diam is even for diam > 1 -- included here for literal agreement
    # with the paper).
    upper = 2 * diam + 1
    if upper <= 3:
        return []
    mstd_primes: List[int] = []
    for p in _primes_up_to(upper):
        mods = [c % p for c in coeffs]
        # Bitmask-based distinctness check: build a bitmask of which residues
        # appear; if popcount < n, two coefficients collided mod p.
        mod_mask = 0
        for m in mods:
            mod_mask |= 1 << m
        if bin(mod_mask).count('1') != n:
            continue
        ss_bits = 0
        ds_bits = 0
        for x in mods:
            for y in mods:
                ss_bits |= 1 << ((x + y) % p)
                ds_bits |= 1 << ((x - y) % p)
        if bin(ss_bits).count('1') > bin(ds_bits).count('1'):
            mstd_primes.append(p)
    return mstd_primes


def check_mstd_integer(cfg: Config) -> bool:
    """Check whether a leaf config is MSTD as an integer set: |A+A| > |A-A|.

    For a leaf config with 1 variable, the coefficients (after GCD-reduce)
    represent an integer set (assuming the variable specializes to any nonzero
    integer; the MSTD property is invariant under that scaling).

    Returns True if the set is MSTD in Z, False otherwise.
    Returns False if cfg isn't a valid leaf or has duplicate coefficients.
    """
    coeffs = _leaf_coeffs(cfg)
    if coeffs is None:
        return False
    if len(set(coeffs)) != len(coeffs):
        return False  # duplicate coeffs -> fewer than |A| distinct elements in Z
    ss: Set[int] = set()
    ds: Set[int] = set()
    for x in coeffs:
        for y in coeffs:
            ss.add(x + y)
            ds.add(x - y)
    return len(ss) > len(ds)


def check_mstd_modp_leaf(cfg: Config, p: int) -> bool:
    """MSTD test for a one-variable leaf in the SINGLE fixed characteristic p.

    Used by beta runs (multivar_prime_filter = p) so a leaf's MSTD-ness is judged
    only in the active field Z/pZ -- never in some other characteristic.  Reduces
    the leaf coefficients mod p; if two collide (so the set has < |A| elements in
    F_p) it is not the intended full-size set and returns False.  Otherwise
    compares |A+A| vs |A-A| in Z/pZ.
    """
    coeffs = _leaf_coeffs(cfg)
    if coeffs is None:
        return False
    mods = [c % p for c in coeffs]
    if len(set(mods)) != len(mods):
        return False
    S: Set[int] = set()
    D: Set[int] = set()
    for x in mods:
        for y in mods:
            S.add((x + y) % p)
            D.add((x - y) % p)
    return len(S) > len(D)


def affine_canonical_form(coeffs: List[int], p: int) -> Tuple[int, ...]:
    """Lex-smallest representative of the affine-equivalence class of the set
    A = {c mod p} under all maps x -> lambda*x + t (lambda in F_p^*, t in F_p).

    This is the finite-field canonicalizer the manuscript's "unique up to affine
    transformation" counts require.  Returns the canonical sorted tuple; two
    finite-field sets are affine-equivalent iff they share this canonical form.
    Cost O(p^2 * |A| log|A|); intended for postprocessing, not the hot path.
    """
    A = sorted({c % p for c in coeffs})
    best: Optional[Tuple[int, ...]] = None
    for lam in range(1, p):
        for t in range(p):
            img = tuple(sorted((lam * a + t) % p for a in A))
            if best is None or img < best:
                best = img
    return best  # type: ignore


# ----------------------------------------------------------------------------
# Analyzer
# ----------------------------------------------------------------------------
class Analyzer:
    """MSTD symbolic analyzer.

    Usage:
        A = Analyzer(['0', 'b-c', 'b', 'c', 'b-c+e', 'e', 'f', 'g', 'h'])
        results = A.run()

    Constraints:
        - At most 9 elements (MAX_ELEMENTS), at most 8 variables (MAX_VARIABLES).
        - All variables are assumed distinct and nonzero in Z/pZ.
        - The element 0 may appear in the config and is treated as the
          additive identity throughout.
    """

    MAX_ELEMENTS = 9      # manuscript Sections 3.1-3.4 cover sizes 6..9
    MAX_VARIABLES = 8

    def __init__(
        self,
        config_strings: List[str],
        status_every_seconds: float = 30.0,
        skip_top_level_keys: Optional[List[str]] = None,
        checkpoint_path: Optional[str] = None,
        checkpoint_every_seconds: float = 300.0,
        feature_prime_reduction: Optional[int] = None,
        multivar_prime_filter: Optional[int] = None,
        store_for_primes_larger_than: Optional[int] = None,
        interchangeable_groups: Optional[List[List[str]]] = None,
    ):
        """
        Parameters
        ----------
        config_strings : list of str
            The starting configuration as strings, e.g.
            ['0', 'b-c', 'b', 'c', 'b-c+e', 'e', 'f', 'g', 'h'].
        status_every_seconds : float
            How often to print the status box.
        skip_top_level_keys : list of str, optional
            Top-level collision keys to skip (e.g. cases you've already
            completed in a previous run).  Strings are matched against the
            canonical Expr representation produced by expr_to_str(key).
        checkpoint_path : str, optional
            If given, periodically pickle the analyzer state to this file.
            Allows recovery if the process dies.  The file is overwritten
            atomically each save.
        checkpoint_every_seconds : float
            How often to save the checkpoint, in seconds.  Only used if
            checkpoint_path is given.
        """
        if len(config_strings) > self.MAX_ELEMENTS:
            raise ValueError(
                f"Config has {len(config_strings)} elements; max is {self.MAX_ELEMENTS}.")
        cfg_unsorted = tuple(parse_expr_from_str(s) for s in config_strings)
        self.initial_config: Config = config_canonical(cfg_unsorted)
        if multivar_prime_filter is not None:
            self.initial_config = config_canonical(
                tuple(expr_mod_p(e, multivar_prime_filter) for e in self.initial_config))
        vars_used = config_vars(self.initial_config)
        if len(vars_used) > self.MAX_VARIABLES:
            raise ValueError(
                f"Config uses {len(vars_used)} variables; max is {self.MAX_VARIABLES}.")
        self.num_vars = len(vars_used)
        self.status_every_seconds = float(status_every_seconds)
        # Feature: when set to an integer p, print every branch whose step
        # reduction factor is divisible by p.  None disables it.
        self.feature_prime_reduction: Optional[int] = feature_prime_reduction
        self._feature_reduction_count: int = 0
        # Stack of (key, gcd) applied from the initial config down to the
        # current node; used to print the key-path when the feature fires.
        self._key_path: List[Tuple[Expr, int]] = []
        # Feature: reduce every coefficient mod p (into {0,...,p-1}) when
        # tallying |A+A| / |A-A| for the multi-variable MSTD check.  None =
        # off (exact generic tally).  When set, expressions live in the free
        # (Z/pZ)-module on the variables: e.g. mod 2, 2a -> 0 and -b -> b.
        self.multivar_prime_filter: Optional[int] = multivar_prime_filter
        # Feature: store_for_primes_larger_than = q.  When not None, for every
        # reduction factor r encountered while processing a top-level case,
        # each distinct prime factor pf of r with pf >= q gets that case's
        # top-level key string recorded in reduction_cases_by_prime[pf].  Uses
        # a set, so a given key never appears twice for the same prime.
        self.store_for_primes_larger_than: Optional[int] = store_for_primes_larger_than
        self.reduction_cases_by_prime: Dict[int, Set[str]] = defaultdict(set)
        # Interchangeability: explicit, caller-justified groups of variable names
        # that are genuinely symmetric in the initial config.  When None/empty,
        # NO interchangeability collapsing is done (fully exhaustive).  Occurrence
        # count is NOT used as an automatic symmetry test anymore.
        self.interchangeable_groups: Optional[List[List[str]]] = interchangeable_groups
        if interchangeable_groups:
            self._validate_interchangeable_groups(interchangeable_groups)
        # Skip-list: top-level collision keys to skip
        self.skip_top_level_key_strs: Set[str] = set(skip_top_level_keys or [])
        # Checkpointing
        self.checkpoint_path: Optional[str] = checkpoint_path
        self.checkpoint_every_seconds: float = float(checkpoint_every_seconds)
        self._t_last_checkpoint: float = 0.0
        # Dedup: keys of configs we've already finished exploring.
        # The keys are canonical forms invariant under variable renaming,
        # produced by config_dedup_key(). This means two configs that
        # differ only in which variables are named what (or which order
        # the elements are listed in) will dedup to the same key.
        self.seen: Set[Tuple] = set()
        # Reduction tracking — both kept cheap:
        # global_max_prime_dividing = max over all steps of
        #   largest_prime_factor(step_reduction)
        # global_max_reduction = max step_reduction itself (informational)
        self.global_max_prime_dividing: int = 1
        self.global_max_reduction: int = 1
        # MSTD storage:
        #   mstd_configs_by_prime[p] = set of leaf Configs that are MSTD in Z/pZ
        #   high_depth_mstd_configs = set of 2-variable Configs whose ENTIRE
        #     set of child collisions (apply_collision results) are MSTD leaves
        #     in some prime (option (b): even duplicate children disqualify).
        self.mstd_configs_by_prime: Dict[int, Set[Config]] = defaultdict(set)
        self.high_depth_mstd_configs: Set[Config] = set()
        # Integer-MSTD leaves: leaf configs (1 variable) whose integer
        # coefficient set is MSTD over Z (not just in some Z/pZ).
        self.mstd_integer_configs: Set[Config] = set()
        # Multi-variable MSTD configs: configs with >= 2 variables remaining
        # that are MSTD (|A+A| > |A-A|) at a generic specialization. Each
        # distinct config is checked at most once (gated by the `seen` set).
        self.multivar_mstd_configs: Set[Config] = set()
        # Stats
        self.branches_processed: int = 0
        self.duplicate_terminations: int = 0
        self.dedup_skipped: int = 0
        # Per-variable-count skip counters (only 2..5 tracked; others lumped in dedup_skipped)
        self.dedup_skipped_by_vars: Dict[int, int] = {2: 0, 3: 0, 4: 0, 5: 0}
        # Time accumulator: total seconds spent on `dkey in seen` lookups.
        self._seen_lookup_time: float = 0.0
        # Status state
        self._t_start: float = 0.0
        self._t_last_status: float = 0.0
        self._current_case: int = 0
        self._total_cases: int = 0
        self._current_subcase: int = 0
        self._total_subcases: int = 0
        self._current_case_key: str = ""
        self._current_subcase_key: str = ""
        # Snapshots at the previous status update (for rate calculations)
        self._snap_branches: int = 0
        self._snap_dups: int = 0
        self._snap_skipped: int = 0
        self._snap_seen: int = 0
        self._snap_time: float = 0.0
        self._snap_skipped_by_vars: Dict[int, int] = {2: 0, 3: 0, 4: 0, 5: 0}
        self._snap_seen_lookup_time: float = 0.0
        self._snap_multivar: int = 0

    # ------------------------------------------------------------------ #
    # Helpers
    # ------------------------------------------------------------------ #
    def _record_step_reduction(self, r: int, cfg: Optional[Config] = None,
                               key: Optional[Expr] = None,
                               new_cfg: Optional[Config] = None) -> None:
        """This method records a single reduction factor into the program's
        running certificate.  It updates the largest reduction seen and the
        largest prime dividing any reduction (phi), and, when those features are
        enabled, it also notes which top-level case produced the factor and prints
        the collision path that first introduced that prime."""
        if r > self.global_max_reduction:
            self.global_max_reduction = r
        if r > 1:
            pf = largest_prime_factor(r)
            if pf > self.global_max_prime_dividing:
                self.global_max_prime_dividing = pf
        # Feature: record which top-level case produced each reduction prime.
        # For every distinct prime factor pf of r with pf >= q, add the current
        # top-level case key to reduction_cases_by_prime[pf].  Guarded on a
        # non-empty case key so reductions from the pre-loop initial-config key
        # scan (which belong to no single case) aren't misattributed.
        q = self.store_for_primes_larger_than
        if q is not None and r > 1 and self._current_case_key:
            for pf in distinct_prime_factors(r):
                if pf >= q:
                    self.reduction_cases_by_prime[pf].add(self._current_case_key)
        # Feature: when applying `key` completes the FIRST collision on this
        # root->here path whose reduction is divisible by fp, print the key
        # path.  self._key_path holds the ancestor keys (the current `key` is
        # not yet pushed), so "no ancestor divisible by fp" == "this is first".
        fp = self.feature_prime_reduction
        if fp and r > 1 and r % fp == 0 and key is not None:
            if not any(g % fp == 0 for _, g in self._key_path):
                self._print_feature_reduction(key, r)

    def _print_feature_reduction(self, key: Expr, r: int) -> None:
        """Print the sequence of collision keys from the initial config down to
        (and including) `key`, the first key on this path whose gcd is divisible
        by self.feature_prime_reduction.  Each key is shown in its raw collision
        form (i.e. multiplied back by its gcd), so the final one visibly carries
        the divisible-by-p gcd."""
        self._feature_reduction_count += 1
        fp = self.feature_prime_reduction
        path = list(self._key_path) + [(key, r)]
        parts = []
        for i, (k, g) in enumerate(path, 1):
            raw = expr_scale(k, g) if g > 1 else k
            parts.append(f"Key {i}: {expr_to_str(raw)}")
        print(f"  ⚙ FEATURE p={fp} | " + ". ".join(parts), flush=True)

    def _get_collision_keys(self, cfg: Config, apply_inter: bool) -> Dict[Expr, int]:
        """This method is the analyzer's wrapper around get_collision_keys.  It
        feeds every reduction factor found in the configuration into the phi
        trackers, and, only at the top level and only when the caller has declared
        interchangeable variable groups, it collapses the returned keys down to one
        representative per symmetry class before handing them back."""
        keys, reduction_factors = get_collision_keys(cfg, self.multivar_prime_filter)
        for factor in reduction_factors:
            # Every raw-collision reduction factor feeds the reduction / max-prime
            # trackers -- including factors from "invalid" collisions (3a=3b -> 3,
            # 17a=17c -> 17) and single-variable coincidences (7b=0 -> 7).
            # key=None so this never triggers the feature-path printout.
            self._record_step_reduction(factor)
        # Interchangeability collapsing happens ONLY when explicit groups were
        # supplied.  With no groups, apply_inter is a no-op -> fully exhaustive.
        if apply_inter and self.interchangeable_groups:
            kept = filter_interchangeable(list(keys), self.interchangeable_groups)
            return {k: keys[k] for k in kept}
        return keys

    @staticmethod
    def _swap_vars_in_config(cfg: Config, a: str, b: str) -> Config:
        """This method returns a copy of the configuration in which the two named
        variables have been exchanged everywhere and the result put back into
        canonical form.  The program uses it to test whether two variables are
        genuinely interchangeable, since a valid symmetry must leave the
        configuration unchanged after the swap."""
        def swap_expr(e: Expr) -> Expr:
            d: Dict[str, int] = {}
            for v, c in e:
                nv = b if v == a else (a if v == b else v)
                d[nv] = d.get(nv, 0) + c
            return expr_from_dict(d)
        return config_canonical(tuple(swap_expr(e) for e in cfg))

    def _validate_interchangeable_groups(self, groups: List[List[str]]) -> None:
        """Reject interchangeable groups that are not genuine symmetries of the
        initial config.  Occurrence count is NOT a symmetry proof, so we verify
        directly: every transposition within a group must leave the initial
        config invariant (transpositions generate the full symmetric group).
        Also checks the groups are disjoint and reference real variables."""
        cfg = self.initial_config
        present = config_vars(cfg)
        seen_vars: Set[str] = set()
        for g in groups:
            for v in g:
                if v not in present:
                    raise ValueError(
                        f"interchangeable group variable {v!r} is not in the "
                        f"config variables {sorted(present)}.")
                if v in seen_vars:
                    raise ValueError(
                        f"variable {v!r} appears in more than one interchangeable "
                        f"group; groups must be disjoint.")
                seen_vars.add(v)
            for i in range(len(g)):
                for j in range(i + 1, len(g)):
                    if self._swap_vars_in_config(cfg, g[i], g[j]) != cfg:
                        raise ValueError(
                            f"variables {g[i]!r} and {g[j]!r} were declared "
                            f"interchangeable, but swapping them changes the "
                            f"initial config -- it is NOT a genuine symmetry.")

    def _record_leaf(self, cfg: Config) -> None:
        """Record MSTD-ness of a <=1-variable leaf, respecting beta mode.

        In a beta run (multivar_prime_filter = p) the leaf is judged ONLY in
        characteristic p (check_mstd_modp_leaf) and the integer check is skipped.
        In a generic (characteristic-zero) run the leaf is checked over Z and
        across the prime range (check_mstd_primes / check_mstd_integer)."""
        p = self.multivar_prime_filter
        if p is not None:
            if check_mstd_modp_leaf(cfg, p):
                self.mstd_configs_by_prime[p].add(cfg)
            return
        for pr in check_mstd_primes(cfg):
            self.mstd_configs_by_prime[pr].add(cfg)
        if check_mstd_integer(cfg):
            if cfg not in self.mstd_integer_configs:
                self.mstd_integer_configs.add(cfg)
                self._print_integer_mstd(cfg)

    def _leaf_is_mstd(self, cfg: Config) -> bool:
        """Whether a leaf is MSTD, in the active characteristic if beta mode is
        on, else in some prime in the generic range.  Used by the high-depth
        2-variable lookahead so a beta=p conclusion isn't supported by a child
        that is MSTD only in some other characteristic."""
        p = self.multivar_prime_filter
        if p is not None:
            return check_mstd_modp_leaf(cfg, p)
        return bool(check_mstd_primes(cfg))

    def affine_classes_by_prime(self) -> Dict[int, Dict[Tuple[int, ...], Config]]:
        """Postprocess mstd_configs_by_prime into finite-field affine classes.

        For each prime p, groups the MSTD leaves by affine_canonical_form (i.e.
        up to x -> lambda*x + t in F_p), keeping one representative Config per
        class.  len(result[p]) is the manuscript's "unique up to affine
        transformation" count for Z/pZ.  Computed on demand (can be slow for
        large p), not during the search."""
        out: Dict[int, Dict[Tuple[int, ...], Config]] = {}
        for p, configs in self.mstd_configs_by_prime.items():
            classes: Dict[Tuple[int, ...], Config] = {}
            for cfg in configs:
                coeffs = _leaf_coeffs(cfg)
                if coeffs is None:
                    continue
                key = affine_canonical_form(coeffs, p)
                classes.setdefault(key, cfg)
            out[p] = classes
        return out

    def _format_eta(self, seconds: float) -> str:
        """This method formats a duration given in seconds as a short, readable
        string in seconds, minutes, or hours for the status display, returning
        '--' for negative or implausibly large values."""
        if seconds < 0 or seconds > 3600 * 999:
            return "--"
        if seconds < 60:
            return f"{seconds:.0f}s"
        m = seconds / 60
        if m < 60:
            return f"{m:.0f}m"
        h = m / 60
        return f"{h:.1f}h"

    @classmethod
    def load_checkpoint(
        cls,
        path: str,
        config_strings: List[str],
        skip_top_level_keys: Optional[List[str]] = None,
        status_every_seconds: float = 30.0,
        checkpoint_every_seconds: float = 300.0,
        multivar_prime_filter=_UNSET,
        feature_prime_reduction=_UNSET,
        store_for_primes_larger_than=_UNSET,
        interchangeable_groups=_UNSET,
    ) -> "Analyzer":
        """Build an Analyzer pre-loaded with state from a checkpoint file.

        Parameters
        ----------
        path : str
            Path to a checkpoint pickle file produced by save_checkpoint().
        config_strings : list of str
            The starting configuration.  MUST match the initial config that
            was in use when the checkpoint was saved; ValueError otherwise.
        skip_top_level_keys : list of str, optional
            Top-level cases to skip in the resumed run.  YOU manage this list
            manually — the analyzer does NOT auto-record completed cases.
            Pass the same list you originally used, plus any cases you've
            since confirmed are complete (e.g., from inspecting the previous
            run's printout).
        status_every_seconds, checkpoint_every_seconds :
            Same as in __init__.

        Returns
        -------
        Analyzer : a new analyzer with:
            - seen set restored
            - mstd_configs_by_prime restored
            - high_depth_mstd_configs restored
            - reduction counters restored
            - skip_top_level_key_strs set ONLY to what you pass in
              (not auto-loaded from the checkpoint).
        """
        import pickle
        with open(path, "rb") as f:
            state = pickle.load(f)

        # Resolve each mode option: if the caller left it unset, take the
        # checkpoint's saved value; if the caller supplied one, it MUST match the
        # checkpoint (else a beta=p run could silently resume in generic mode, or
        # vice versa).  Resolving BEFORE construction is essential because
        # multivar_prime_filter changes how initial_config is built.
        def _resolve(name, user_val):
            saved = state.get(name, None)
            if user_val is _UNSET:
                return saved
            if user_val != saved:
                raise ValueError(
                    f"Resume option {name}={user_val!r} does not match the "
                    f"checkpoint's value {saved!r}. Omit it to use the "
                    f"checkpoint's setting, or pass the matching value.")
            return user_val

        mpf = _resolve('multivar_prime_filter', multivar_prime_filter)
        fpr = _resolve('feature_prime_reduction', feature_prime_reduction)
        sfp = _resolve('store_for_primes_larger_than', store_for_primes_larger_than)
        ig = _resolve('interchangeable_groups', interchangeable_groups)

        # Build a fresh analyzer with the SAME mode options (so initial_config,
        # interchangeability, etc. reconstruct identically) and the user's
        # explicit skip list (NOT the checkpoint's saved one).
        a = cls(
            config_strings=config_strings,
            status_every_seconds=status_every_seconds,
            skip_top_level_keys=skip_top_level_keys,
            checkpoint_path=path,
            checkpoint_every_seconds=checkpoint_every_seconds,
            feature_prime_reduction=fpr,
            multivar_prime_filter=mpf,
            store_for_primes_larger_than=sfp,
            interchangeable_groups=ig,
        )
        # Verify initial config matches
        if state['initial_config'] != a.initial_config:
            raise ValueError(
                "Initial config in checkpoint does not match config_strings.\n"
                f"  checkpoint: {state['initial_config']}\n"
                f"  given:      {a.initial_config}"
            )
        # Restore state
        a.seen = state['seen']
        a.mstd_configs_by_prime = defaultdict(set)
        for p, configs in state['mstd_configs_by_prime'].items():
            a.mstd_configs_by_prime[p] = set(configs)
        a.high_depth_mstd_configs = set(state['high_depth_mstd_configs'])
        # Reduction-case tracking may be missing from older checkpoints
        a.reduction_cases_by_prime = defaultdict(set)
        for p, v in state.get('reduction_cases_by_prime', {}).items():
            a.reduction_cases_by_prime[p] = set(v)
        # Integer-MSTD set may be missing from older checkpoints
        a.mstd_integer_configs = set(state.get('mstd_integer_configs', set()))
        # Multi-var-MSTD set may be missing from older checkpoints
        a.multivar_mstd_configs = set(state.get('multivar_mstd_configs', set()))
        a.global_max_reduction = state['global_max_reduction']
        a.global_max_prime_dividing = state['global_max_prime_dividing']
        a.branches_processed = state['branches_processed']
        a.duplicate_terminations = state['duplicate_terminations']
        a.dedup_skipped = state['dedup_skipped']
        total_mstd = sum(len(v) for v in a.mstd_configs_by_prime.values())
        print(f"Loaded checkpoint from {path}:")
        print(f"  seen-set size:           {len(a.seen):,}")
        print(f"  skip-list (user-given):  {len(a.skip_top_level_key_strs)}")
        print(f"  MSTD configs restored:   {total_mstd:,}")
        print(f"  high-depth MSTD:         {len(a.high_depth_mstd_configs):,}")
        print(f"  integer MSTD configs:    {len(a.mstd_integer_configs):,}")
        print(f"  multi-var MSTD configs:  {len(a.multivar_mstd_configs):,}")
        print(f"  max_reduction / prime:   {a.global_max_reduction} / "
              f"{a.global_max_prime_dividing}")
        return a

    def purge_low_var_from_seen(self, max_vars_to_purge: int = 2) -> int:
        """Remove all entries from seen whose underlying config has
        ≤ max_vars_to_purge variables.

        Identification: each inner tuple in a dedup_key has length equal to
        the variable count of the config (for k>=1).  So a 1-var key has
        all-length-1 inner tuples; a 2-var key has all-length-2 inner tuples.

        Returns the number of entries removed.

        Safe to call any time, including before a checkpoint save.  After
        purging, the purged configs may be re-explored on resume — but
        the small ones are cheap (1-var leaves are O(1), 2-var configs do
        one level of recursion to leaves).  The pickle / memory savings
        usually dominate.
        """
        def key_var_count(k) -> int:
            if not k:
                return 0
            # All inner tuples have the same length == variable count (for k>=1)
            first = k[0]
            if isinstance(first, tuple):
                return len(first)
            return 0

        n_before = len(self.seen)
        self.seen = {k for k in self.seen if key_var_count(k) > max_vars_to_purge}
        return n_before - len(self.seen)

    def save_checkpoint(self, path: Optional[str] = None, background: bool = False) -> str:
        """Pickle the current analyzer state to disk.

        Saves: seen set, MSTD results, high-depth MSTD configs, reduction
        trackers, branch/dup/skip counts, and the original initial config.

        Atomic write: pickles to <path>.tmp then renames over <path>, so
        a crash during the save can't corrupt an existing checkpoint.

        Parameters
        ----------
        path : str, optional
            Override the default checkpoint_path.
        background : bool, default False
            If True, snapshot the state quickly and pickle in a background
            thread, returning immediately.  Use for periodic saves to avoid
            blocking the search.  Use False (default) for final saves where
            we want to know it's actually written.

        Returns
        -------
        str : the path that will be written to (may not be written yet if
              background=True).
        """
        import pickle, os, threading
        if path is None:
            path = self.checkpoint_path
        if path is None:
            raise ValueError("No checkpoint_path set on analyzer and no path given.")
        # If a background save is already in flight, wait for it to finish
        # before starting another (avoid disk thrashing and racing saves).
        if hasattr(self, '_save_thread') and self._save_thread is not None:
            if self._save_thread.is_alive():
                self._save_thread.join()
        # SNAPSHOT phase (blocks search but is fast — just shallow copies):
        # Sets and dicts are shallow-copied so the background thread can
        # safely pickle them while the search keeps mutating the originals.
        # Tuple contents are immutable so no deep copy needed.
        state = {
            'initial_config': self.initial_config,
            'seen': set(self.seen),  # snapshot
            'mstd_configs_by_prime': {p: set(v) for p, v in self.mstd_configs_by_prime.items()},
            'high_depth_mstd_configs': set(self.high_depth_mstd_configs),
            'mstd_integer_configs': set(self.mstd_integer_configs),
            'multivar_mstd_configs': set(self.multivar_mstd_configs),
            'global_max_reduction': self.global_max_reduction,
            'global_max_prime_dividing': self.global_max_prime_dividing,
            'branches_processed': self.branches_processed,
            'duplicate_terminations': self.duplicate_terminations,
            'dedup_skipped': self.dedup_skipped,
            'reduction_cases_by_prime': {p: set(v) for p, v in self.reduction_cases_by_prime.items()},
            # Mode options -- saved so a resume can restore them (and validate
            # against any user-supplied overrides) instead of silently reverting
            # to generic mode.  Critical for long beta=p runs.
            'multivar_prime_filter': self.multivar_prime_filter,
            'feature_prime_reduction': self.feature_prime_reduction,
            'store_for_primes_larger_than': self.store_for_primes_larger_than,
            'interchangeable_groups': self.interchangeable_groups,
        }
        tmp_path = path + ".tmp"

        def _do_save():
            try:
                with open(tmp_path, "wb") as f:
                    pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)
                os.replace(tmp_path, path)
            except Exception as e:
                # Background save errors get printed but don't crash the main thread
                print(f"  [WARN: background checkpoint save failed: {e}]", flush=True)

        if background:
            self._save_thread = threading.Thread(target=_do_save, daemon=True)
            self._save_thread.start()
        else:
            _do_save()
        return path

    def _maybe_checkpoint(self) -> None:
        """This method saves a checkpoint in the background, but only if enough
        time has passed since the last one, so that a long run can be resumed
        after a crash without slowing the search down with frequent saves."""
        if self.checkpoint_path is None:
            return
        now = time.time()
        if now - self._t_last_checkpoint < self.checkpoint_every_seconds:
            return
        self._t_last_checkpoint = now
        try:
            self.save_checkpoint(background=True)
        except Exception as e:
            # Don't let a checkpoint failure kill the run; just warn.
            print(f"  [WARN: checkpoint save failed: {e}]", flush=True)

    def _print_integer_mstd(self, cfg: Config) -> None:
        """Print a newly-found integer MSTD leaf as a sorted integer set,
        along with |A+A|, |A-A|, and the diameter.

        Called only on first addition (callers check the set before calling).
        """
        coeffs = _leaf_coeffs(cfg)
        if coeffs is None:
            return
        coeffs_sorted = sorted(coeffs)
        # Shift so min is 0 (canonical translate form)
        m = coeffs_sorted[0]
        shifted = [c - m for c in coeffs_sorted]
        # Compute |A+A|, |A-A|
        S: Set[int] = set()
        D: Set[int] = set()
        for x in shifted:
            for y in shifted:
                S.add(x + y)
                D.add(x - y)
        diam = shifted[-1]
        count_so_far = len(self.mstd_integer_configs)
        print(
            f"  ★ INTEGER MSTD #{count_so_far}: {shifted}  "
            f"|A+A|={len(S)}, |A-A|={len(D)}, diam={diam}",
            flush=True,
        )

    def _symbolic_sumdiff_sizes(self, cfg: Config) -> Tuple[int, int]:
        """Return (|A+A|, |A-A|) for cfg at a GENERIC specialization of its
        variables — i.e. treating them as algebraically independent.

        If self.multivar_prime_filter is set to a prime p, every coefficient is
        first reduced mod p into {0, ..., p-1} (variables stay formal), so
        expressions that coincide mod p are merged before A+A / A-A are tallied.
        See _symbolic_sumdiff_sizes_modp.

        Method: assign variable i the value B**(i+1) for a base B chosen
        larger than twice the largest coefficient magnitude that can occur in
        any pairwise sum or difference. With that base the positional
        (base-B) encoding is faithful: two sums (or two differences) evaluate
        to the same integer iff their symbolic coefficient vectors are equal.
        So the integer |A+A|, |A-A| computed here equal the exact generic
        symbolic values (no false collisions, no missed ones).
        """
        if self.multivar_prime_filter is not None:
            return self._symbolic_sumdiff_sizes_modp(cfg, self.multivar_prime_filter)
        vars_used = sorted(config_vars(cfg))
        k = len(vars_used)
        if k == 0:
            return (0, 0)
        maxc = 0
        for e in cfg:
            for _, c in e:
                ac = c if c >= 0 else -c
                if ac > maxc:
                    maxc = ac
        # Sums/diffs of two elements have coeffs in [-2maxc, 2maxc]; comparing
        # two of them gives digit-differences up to 4maxc, so B > 4maxc+1
        # guarantees a faithful encoding. Use 4maxc+3 for margin.
        B = 4 * maxc + 3
        vals = {v: B ** (i + 1) for i, v in enumerate(vars_used)}
        pts = [sum(c * vals[v] for v, c in e) for e in cfg]
        S: Set[int] = set()
        D: Set[int] = set()
        for x in pts:
            for y in pts:
                S.add(x + y)
                D.add(x - y)
        return (len(S), len(D))

    @staticmethod
    def _reduce_modp(coeffs: Dict[str, int], p: int) -> Expr:
        """Canonicalize a coefficient dict mod p: each coeff -> its residue in
        {0,...,p-1}, dropping any that reduce to 0.  Two expressions are equal
        mod p iff they produce the same tuple."""
        return tuple(sorted((v, c % p) for v, c in coeffs.items() if c % p != 0))

    def _symbolic_sumdiff_sizes_modp(self, cfg: Config, p: int) -> Tuple[int, int]:
        """(|A+A|, |A-A|) with all coefficients reduced mod p into {0,...,p-1}.

        The variables remain formal and independent; only their integer
        coefficients are taken mod p.  Elements that become equal after
        reduction are merged, so A is the set of distinct reduced elements.
        Every pairwise sum and difference is likewise reduced mod p (so e.g.
        -b becomes (p-1)*b, and 2a becomes 0 when p = 2) before being counted.
        """
        red = self._reduce_modp
        elems = list({red(dict(e), p) for e in cfg})
        S: Set[Expr] = set()
        D: Set[Expr] = set()
        for x in elems:
            dx = dict(x)
            for y in elems:
                sd = dict(dx)
                dd = dict(dx)
                for v, c in y:
                    sd[v] = sd.get(v, 0) + c
                    dd[v] = dd.get(v, 0) - c
                S.add(red(sd, p))
                D.add(red(dd, p))
        return (len(S), len(D))

    def _is_mstd_symbolic(self, cfg: Config) -> bool:
        """True iff cfg is MSTD (|A+A| > |A-A|) at a generic specialization."""
        s, d = self._symbolic_sumdiff_sizes(cfg)
        return s > d

    def _print_multivar_mstd(self, cfg: Config) -> None:
        """This method prints a newly discovered multi-variable MSTD configuration,
        listing the variables it depends on together with its sumset and difference
        set sizes, so the operator can see each genuine symbolic MSTD family as the
        search uncovers it."""
        vs = sorted(config_vars(cfg))
        s, d = self._symbolic_sumdiff_sizes(cfg)
        count = len(self.multivar_mstd_configs)
        print(
            f"  ◆ MULTI-VAR MSTD #{count} ({len(vs)} vars {vs}): "
            f"{config_to_str(cfg)}  |A+A|={s}, |A-A|={d}",
            flush=True,
        )

    def _print_status_box(self) -> None:
        """This method prints the multi-line status box that reports the search's
        live progress -- elapsed time, current case and subcase, nodes explored,
        leaves found, and the running phi certificate -- laid out as a small table
        so a long run can be monitored at a glance."""
        now = time.time()
        elapsed = now - self._t_start
        # Compute deltas since last status (over the past ~`status_every_seconds`)
        delta_time = now - self._snap_time if self._snap_time else elapsed
        delta_branches = self.branches_processed - self._snap_branches
        delta_dups = self.duplicate_terminations - self._snap_dups
        delta_skipped = self.dedup_skipped - self._snap_skipped
        seen_size = len(self.seen)
        delta_seen = seen_size - self._snap_seen
        delta_skipped_by_vars = {
            k: self.dedup_skipped_by_vars[k] - self._snap_skipped_by_vars[k]
            for k in (2, 3, 4, 5)
        }
        delta_seen_lookup_time = self._seen_lookup_time - self._snap_seen_lookup_time
        eta_str = "--"
        if self._total_cases and self._current_case > 0 and elapsed > 0:
            rate = self._current_case / elapsed
            eta_str = self._format_eta((self._total_cases - self._current_case) / max(rate, 1e-9))
        elapsed_str = self._format_eta(elapsed)
        delta_label = f"~{delta_time:.0f}s"
        W = 70  # inner width of box content (between the two │ borders)
        # Top border with embedded title and ETA
        title = f" STATUS @ {elapsed_str} "
        eta_label = f" ETA: {eta_str} "
        dash_count = W - len(title) - len(eta_label)
        if dash_count < 0:
            dash_count = 0
        top_line = "┌" + title + ("─" * dash_count) + eta_label + "┐"
        # Body
        body: List[str] = []
        body.append(f"Case  {self._current_case:>4} / {self._total_cases:<4}")
        if self._current_case_key:
            body.append(f"  └─ key:  {self._current_case_key}")
        if self._total_subcases:
            body.append(f"Subcase  {self._current_subcase:>4} / {self._total_subcases:<4}")
            if self._current_subcase_key:
                body.append(f"  └─ key:  {self._current_subcase_key}")
        body.append("")
        # Cumulative totals
        body.append("CUMULATIVE")
        body.append(
            f"  Branches: {self.branches_processed:>13,}   "
            f"Skipped: {self.dedup_skipped:>13,}"
        )
        body.append(
            f"  Dups:     {self.duplicate_terminations:>13,}   "
            f"Seen:    {seen_size:>13,}"
        )
        if self.branches_processed > 0:
            skip_pct = 100.0 * self.dedup_skipped / self.branches_processed
            body.append(f"  Skipped / branches: {skip_pct:>5.1f}%")
        body.append("")
        # Past-30s rates
        body.append(f"PAST {delta_label}")
        body.append(
            f"  Branches: {delta_branches:>13,}   "
            f"Skipped: {delta_skipped:>13,}"
        )
        body.append(
            f"  Dups:     {delta_dups:>13,}   "
            f"+Seen:   {delta_seen:>13,}"
        )
        if delta_branches > 0:
            delta_skip_pct = 100.0 * delta_skipped / delta_branches
            body.append(f"  Skipped / branches: {delta_skip_pct:>5.1f}%")
        # Per-variable-count skip breakdown (just past window).
        body.append(
            f"  Skipped @ 2-var: {delta_skipped_by_vars[2]:>10,}   "
            f"@ 3-var: {delta_skipped_by_vars[3]:>10,}"
        )
        body.append(
            f"  Skipped @ 4-var: {delta_skipped_by_vars[4]:>10,}   "
            f"@ 5-var: {delta_skipped_by_vars[5]:>10,}"
        )
        # Time spent on `seen` lookups in the past window, as % of elapsed.
        if delta_time > 0:
            seen_pct = 100.0 * delta_seen_lookup_time / delta_time
            body.append(
                f"  Time on `seen` lookups: {delta_seen_lookup_time:.2f}s "
                f"({seen_pct:.1f}% of past {delta_time:.0f}s)"
            )
        body.append("")
        # Reduction tracking + MSTD counts
        body.append(
            f"Max red: {self.global_max_reduction:>5}   "
            f"Max prime: {self.global_max_prime_dividing:>4}"
        )
        total_mstd = sum(len(v) for v in self.mstd_configs_by_prime.values())
        body.append(
            f"MSTD leaves: {total_mstd:>8,}   "
            f"High-depth MSTD: {len(self.high_depth_mstd_configs):>5,}"
        )
        delta_multivar = len(self.multivar_mstd_configs) - self._snap_multivar
        body.append(
            f"Integer MSTD: {len(self.mstd_integer_configs):>7,}   "
            f"Multi-var MSTD: {len(self.multivar_mstd_configs):>7,}"
        )
        body.append(
            f"Multi-var MSTD (past {delta_label}): {delta_multivar:>+6,}"
        )
        body.append("")
        # Leaderboard (ranked)
        lb_items = sorted(
            ((p, len(v)) for p, v in self.mstd_configs_by_prime.items()),
            key=lambda x: -x[1])[:8]
        if lb_items:
            body.append("MSTD leaderboard:")
            for rank, (p, c) in enumerate(lb_items, 1):
                body.append(f"  {rank:>2}. Z_{p}: {c:>7,}")
        # Render with right border
        inner_w = W - 2
        lines = [top_line]
        for s in body:
            if len(s) > inner_w:
                s = s[:inner_w - 1] + "…"
            lines.append("│ " + s.ljust(inner_w) + " │")
        lines.append("└" + ("─" * W) + "┘")
        for ln in lines:
            print(ln, flush=True)
        # Update snapshots for next delta
        self._snap_time = now
        self._snap_branches = self.branches_processed
        self._snap_dups = self.duplicate_terminations
        self._snap_skipped = self.dedup_skipped
        self._snap_seen = seen_size
        self._snap_skipped_by_vars = dict(self.dedup_skipped_by_vars)
        self._snap_seen_lookup_time = self._seen_lookup_time
        self._snap_multivar = len(self.multivar_mstd_configs)

    def _maybe_status(self) -> None:
        """This method prints the live status box on a regular wall-clock schedule
        (and opportunistically triggers a checkpoint at the same time).  It is
        called at the top of every recursive step so progress is reported steadily
        no matter how deep the search currently is."""
        now = time.time()
        if now - self._t_last_status < self.status_every_seconds:
            return
        self._t_last_status = now
        self._print_status_box()
        # Opportunistic checkpoint at the same cadence (the checkpoint method
        # has its own time guard so it won't save more often than configured).
        self._maybe_checkpoint()

    # ------------------------------------------------------------------ #
    # Core recursion
    # ------------------------------------------------------------------ #
    def explore(self, cfg: Config, depth: int, max_depth: int) -> None:
        """Recursively explore cfg. cfg MUST be canonical.

        Termination order (cheap checks first):
          1. Duplicate elements -> kill.
          2. Already-explored config -> skip (incl. previously-visited leaves).
          3. <= 1 variable remaining -> leaf diameter record.
          4. Max depth reached -> stop.
        """
        self._maybe_status()
        # 1. Duplicate-element termination (cheap, do first).
        if config_has_duplicates(cfg):
            self.duplicate_terminations += 1
            return
        # 2. Dedup (applies at every depth, including leaves).
        # IMPORTANT: we add to `seen` only AFTER the subtree under cfg is
        # fully explored, so a Ctrl-C / kill mid-exploration doesn't leave
        # partially-explored configs marked as "done" in seen.
        dkey = config_dedup_key(cfg)
        _t = time.perf_counter()
        in_seen = dkey in self.seen
        self._seen_lookup_time += time.perf_counter() - _t
        if in_seen:
            self.dedup_skipped += 1
            # Bucket by var count for the status display
            n = len(dkey[0]) if dkey else 0
            if n in self.dedup_skipped_by_vars:
                self.dedup_skipped_by_vars[n] += 1
            return
        # 3. Leaf? (1-var or 0-var configs)
        vars_left = config_vars(cfg)
        if len(vars_left) <= 1:
            # Beta-aware leaf recording (fixed characteristic vs generic range).
            self._record_leaf(cfg)
            # Leaf is "fully explored" (no children).
            self.seen.add(dkey)
            return
        # 3b. Multi-variable MSTD check (>= 2 vars). Reached only when cfg is
        # NOT in `seen`; the subtree below marks cfg seen, so with the default
        # max_depth each distinct config is checked exactly once.
        if self._is_mstd_symbolic(cfg):
            if cfg not in self.multivar_mstd_configs:
                self.multivar_mstd_configs.add(cfg)
                self._print_multivar_mstd(cfg)
        # 4. Depth guard.
        if depth >= max_depth:
            # Don't add to seen — we didn't fully explore this subtree.
            return
        # 5. High-depth MSTD check: at a config with exactly 2 variables left,
        # check if EVERY child collision produces an MSTD leaf (strict: even
        # duplicate-element children disqualify the flag).  Doing this here
        # lets us cache the child results for the recursive expansion below.
        # We record the lookahead results in a dict so we don't redo
        # apply_collision twice per child.
        keys = self._get_collision_keys(cfg, apply_inter=False)  # dict key->gcd
        is_2var = (len(vars_left) == 2)
        precomputed_children: Optional[Dict[Expr, Optional[Config]]] = None
        if is_2var:
            precomputed_children = {}
            all_children_mstd = True
            if not keys:
                all_children_mstd = False  # no children => not "all MSTD"
            for key in keys:
                new_cfg, _ = apply_collision(cfg, key, self.multivar_prime_filter)   # ignore config-scale gcd
                precomputed_children[key] = new_cfg
                if new_cfg is None:
                    all_children_mstd = False
                    continue
                cc = config_canonical(new_cfg)
                if config_has_duplicates(cc):
                    all_children_mstd = False
                    continue
                # cc should be a 1-var leaf (since we just eliminated one of
                # two remaining variables).  In beta mode this tests ONLY the
                # active characteristic, so a child MSTD in a different prime
                # cannot support a fixed-characteristic high-depth conclusion.
                if self._leaf_is_mstd(cc):
                    continue  # this child is MSTD (in the active characteristic)
                else:
                    all_children_mstd = False
            if all_children_mstd:
                self.high_depth_mstd_configs.add(cfg)
        # 6. Branch on each collision key.  The reduction recorded is the
        # COLLISION gcd (kgcd), not the config-coefficient gcd.
        for key, kgcd in keys.items():
            self.branches_processed += 1
            if precomputed_children is not None and key in precomputed_children:
                new_cfg = precomputed_children[key]
            else:
                new_cfg, _ = apply_collision(cfg, key, self.multivar_prime_filter)
            if new_cfg is None:
                continue
            # self._key_path currently holds the ancestors of `cfg`; record
            # (which may fire the feature) sees the correct ancestor list.
            self._record_step_reduction(kgcd, cfg=cfg, key=key, new_cfg=new_cfg)
            self._key_path.append((key, kgcd))
            self.explore(config_canonical(new_cfg), depth + 1, max_depth)
            self._key_path.pop()
        # 7. Subtree fully explored — NOW mark this config as seen.
        # If we're killed before reaching this line, the config stays
        # un-seen, so resume will re-explore it correctly.
        self.seen.add(dkey)

    # ------------------------------------------------------------------ #
    # Top-level driver
    # ------------------------------------------------------------------ #
    def print_random_example_branch(self, seed: Optional[int] = None) -> None:
        """Print one random branch from initial config down to a leaf (or until
        the branch terminates by duplicates / no-keys / depth-limit).

        Useful sanity check: shows the user one concrete trace through the
        recursion so they can verify the collision-application logic.

        seed: if given, makes the example reproducible.
        """
        import random
        if seed is not None:
            random.seed(seed)
        print()
        print("=" * 80)
        print("RANDOM EXAMPLE BRANCH (for sanity-checking)")
        print("=" * 80)
        current = self.initial_config
        depth = 0
        max_depth = self.num_vars
        path: List[Expr] = []
        max_red_seen = 1
        while depth < max_depth:
            print(f"--- depth {depth} ---")
            print(f"  config: {config_to_str(current)}")
            vars_left = config_vars(current)
            print(f"  vars left: {sorted(vars_left)} (count={len(vars_left)})")
            if config_has_duplicates(current):
                print(f"  TERMINATED: duplicate elements")
                break
            if len(vars_left) <= 1:
                print(f"  REACHED LEAF (<=1 variable left).")
                mstd_primes = check_mstd_primes(current)
                if mstd_primes:
                    print(f"  MSTD in primes: {mstd_primes}")
                else:
                    print(f"  Not MSTD in any prime in range.")
                break
            keys_d, reduction_factors = get_collision_keys(current, self.multivar_prime_filter)   # dict key -> gcd
            if reduction_factors:
                print(f"  reduction factors (recorded regardless of validity): "
                      f"{sorted(reduction_factors)} "
                      f"(max prime factor {max(largest_prime_factor(c) for c in reduction_factors)})")
            if depth == 0 and self.interchangeable_groups:
                keylist = filter_interchangeable(list(keys_d), self.interchangeable_groups)
            else:
                keylist = list(keys_d)
            print(f"  collision keys available: {len(keylist)}")
            if not keylist:
                print(f"  TERMINATED: no valid collision keys")
                break
            chosen = random.choice(keylist)
            g = keys_d[chosen]   # collision reduction for this key
            print(f"  chose key: {expr_to_str(chosen)}")
            new_cfg, _ = apply_collision(current, chosen, self.multivar_prime_filter)
            if new_cfg is None:
                print(f"  TERMINATED: apply_collision failed")
                break
            if g > 1:
                print(f"  collision reduction = {g} (largest prime factor "
                      f"{largest_prime_factor(g)})")
            max_red_seen = max(max_red_seen, g)
            path.append(chosen)
            current = config_canonical(new_cfg)
            depth += 1
        else:
            print(f"  Reached max_depth = {max_depth}.")
        print()
        print(f"Path: {' -> '.join(expr_to_str(k) for k in path) if path else '(empty)'}")
        print(f"Max reduction factor along path: {max_red_seen}")
        print(f"Final config: {config_to_str(current)}")
        print("=" * 80, flush=True)

    def run(self, max_depth: Optional[int] = None) -> Dict[int, Set[Config]]:
        """This method is the top-level driver of the whole search.  It computes
        the initial configuration's collision keys, then works through each one as
        a separate case, branching on every collision beneath it until the search
        is exhausted; the first level is handled here (so the status box can show a
        case-by-case counter) and everything deeper is delegated to explore().  It
        returns the aggregated results -- the MSTD sets found and the phi
        certificate -- and, if configured, writes a final checkpoint on the way out."""
        if max_depth is None:
            # Recurse until at most 1 variable remains.
            max_depth = self.num_vars
        self._t_start = time.time()
        self._t_last_status = self._t_start
        top_keys_all = self._get_collision_keys(self.initial_config, apply_inter=True)
        # Apply skip-list: drop top-level keys whose string form matches an
        # entry in self.skip_top_level_key_strs (these are cases already
        # completed in a previous run).  Warn about any skip-strings that
        # didn't match any actual key (likely typos or canonical-form
        # mismatches).
        if self.skip_top_level_key_strs:
            top_keys = []
            matched_skip_strs: Set[str] = set()
            for k in top_keys_all:
                s = expr_to_str(k)
                if s in self.skip_top_level_key_strs:
                    matched_skip_strs.add(s)
                else:
                    top_keys.append(k)
            unmatched = self.skip_top_level_key_strs - matched_skip_strs
            print(f"Skip list: {len(matched_skip_strs)} matched, "
                  f"skipping those top-level cases.")
            if unmatched:
                print(f"WARNING: {len(unmatched)} skip-strings did not match any "
                      f"actual top-level key. Likely typos or canonical-form "
                      f"mismatches. Unmatched: {sorted(unmatched)}")
        else:
            top_keys = list(top_keys_all)
        self._total_cases = len(top_keys)
        print("=" * 80)
        print(f"Initial config: {config_to_str(self.initial_config)}")
        print(f"Variables: {sorted(config_vars(self.initial_config))}")
        print(f"Top-level collision keys (after interchangeability): {len(top_keys_all)}")
        if self.skip_top_level_key_strs:
            print(f"Cases to process after skip-list:              {len(top_keys)}")
        print(f"Max depth: {max_depth}")
        if self.checkpoint_path:
            print(f"Checkpoint:  every {self.checkpoint_every_seconds:.0f}s to "
                  f"{self.checkpoint_path}")
        print("=" * 80, flush=True)
        # Print a random example branch for sanity-checking BEFORE starting
        # the full exhaustive run. Uses its own RNG state; doesn't affect run.
        self.print_random_example_branch()
        print(flush=True)
        print("Starting exhaustive search.", flush=True)
        print(flush=True)
        # Multi-variable MSTD check on the initial config (once).
        if len(config_vars(self.initial_config)) >= 2 and \
                self._is_mstd_symbolic(self.initial_config):
            self.multivar_mstd_configs.add(self.initial_config)
            self._print_multivar_mstd(self.initial_config)
        # Mark the initial config as seen up front so it isn't re-entered.
        self.seen.add(config_dedup_key(self.initial_config))
        try:
            for ci, key in enumerate(top_keys, 1):
                self._current_case = ci
                self._current_case_key = expr_to_str(key)
                self._current_subcase = 0
                self._total_subcases = 0
                self._current_subcase_key = ""
                t_case = time.time()
                self.branches_processed += 1
                new_cfg, _ = apply_collision(self.initial_config, key, self.multivar_prime_filter)
                if new_cfg is None:
                    case_time = time.time() - t_case
                    elapsed = time.time() - self._t_start
                    total_mstd = sum(len(v) for v in self.mstd_configs_by_prime.values())
                    print(
                        f"  ⊘  Case {ci:>3}/{self._total_cases:<3}  "
                        f"{expr_to_str(key):<28}  (skipped)  "
                        f"total {self._format_eta(elapsed)}  MSTD {total_mstd:,}",
                        flush=True,
                    )
                    continue
                self._key_path = []   # top-level: no ancestors above this key
                self._record_step_reduction(top_keys_all[key], cfg=self.initial_config, key=key, new_cfg=new_cfg)
                self._key_path = [(key, top_keys_all[key])]   # descend under it
                new_cfg_canon = config_canonical(new_cfg)
                # Inline the first level of explore() so we can track subcases.
                if config_has_duplicates(new_cfg_canon):
                    self.duplicate_terminations += 1
                else:
                    new_cfg_dkey = config_dedup_key(new_cfg_canon)
                    _t = time.perf_counter()
                    in_seen = new_cfg_dkey in self.seen
                    self._seen_lookup_time += time.perf_counter() - _t
                    if in_seen:
                        self.dedup_skipped += 1
                        n = len(new_cfg_dkey[0]) if new_cfg_dkey else 0
                        if n in self.dedup_skipped_by_vars:
                            self.dedup_skipped_by_vars[n] += 1
                    else:
                        # NOTE: add to seen only AFTER full subtree exploration.
                        vars_left = config_vars(new_cfg_canon)
                        # Multi-variable MSTD check (>= 2 vars), once per config.
                        if len(vars_left) >= 2 and self._is_mstd_symbolic(new_cfg_canon):
                            if new_cfg_canon not in self.multivar_mstd_configs:
                                self.multivar_mstd_configs.add(new_cfg_canon)
                                self._print_multivar_mstd(new_cfg_canon)
                        if len(vars_left) <= 1:
                            # Beta-aware leaf recording (see _record_leaf).
                            self._record_leaf(new_cfg_canon)
                        elif max_depth >= 2:
                            sub_keys = self._get_collision_keys(new_cfg_canon, apply_inter=False)
                            self._total_subcases = len(sub_keys)
                            for si, (sub_key, sub_g) in enumerate(sub_keys.items(), 1):
                                self._current_subcase = si
                                self._current_subcase_key = expr_to_str(sub_key)
                                self.branches_processed += 1
                                sub_new_cfg, _ = apply_collision(new_cfg_canon, sub_key, self.multivar_prime_filter)
                                if sub_new_cfg is None:
                                    continue
                                self._record_step_reduction(sub_g, cfg=new_cfg_canon, key=sub_key, new_cfg=sub_new_cfg)
                                self._key_path.append((sub_key, sub_g))
                                self.explore(
                                    config_canonical(sub_new_cfg),
                                    depth=2,
                                    max_depth=max_depth,
                                )
                                self._key_path.pop()
                        # Subtree fully explored -> mark seen.
                        self.seen.add(new_cfg_dkey)
                case_time = time.time() - t_case
                elapsed = time.time() - self._t_start
                total_mstd = sum(len(v) for v in self.mstd_configs_by_prime.values())
                print(
                    f"  ✓  Case {ci:>3}/{self._total_cases:<3}  "
                    f"{expr_to_str(key):<28}  "
                    f"{self._total_subcases:>4} subs  "
                    f"case {self._format_eta(case_time)}  "
                    f"total {self._format_eta(elapsed)}  "
                    f"MSTD {total_mstd:,}",
                    flush=True,
                )
        except KeyboardInterrupt:
            print()
            print("=" * 80)
            print(f"KeyboardInterrupt received after case {self._current_case}/{self._total_cases}.")
            # Purge 1- and 2-variable entries from seen before saving — they
            # bloat the file but are cheap to re-derive on resume.
            n_before = len(self.seen)
            removed = self.purge_low_var_from_seen(max_vars_to_purge=2)
            if removed > 0:
                print(f"Purged {removed:,} entries (≤2 vars) from seen "
                      f"({n_before:,} -> {len(self.seen):,}).")
            if self.checkpoint_path:
                try:
                    written = self.save_checkpoint()
                    print(f"Final checkpoint saved to: {written}")
                except Exception as e:
                    print(f"WARNING: final checkpoint save failed: {e}")
            else:
                print("(No checkpoint_path was configured; state was NOT saved.)")
            print("=" * 80, flush=True)
            raise
        # Successful completion: do one last checkpoint write.
        if self.checkpoint_path:
            try:
                self.save_checkpoint()
            except Exception as e:
                print(f"WARNING: final checkpoint save failed: {e}")
        elapsed = time.time() - self._t_start
        total_mstd = sum(len(v) for v in self.mstd_configs_by_prime.values())
        print("=" * 80)
        print(f"DONE in {elapsed:.1f}s")
        print(f"Branches processed:     {self.branches_processed:,}")
        print(f"Duplicate terminations: {self.duplicate_terminations:,}")
        print(f"Dedup-skipped:          {self.dedup_skipped:,}")
        print(f"Max reduction factor:   {self.global_max_reduction}")
        print(f"Max prime dividing any reduction: {self.global_max_prime_dividing}")
        print(f"Total MSTD leaves found:        {total_mstd:,}")
        print(f"High-depth MSTD configs found:  {len(self.high_depth_mstd_configs):,}")
        print(f"Integer-MSTD leaves found:      {len(self.mstd_integer_configs):,}")
        print(f"Multi-var MSTD configs found:   {len(self.multivar_mstd_configs):,}")
        if self.mstd_configs_by_prime:
            print("MSTD breakdown by prime:")
            for p in sorted(self.mstd_configs_by_prime):
                print(f"  Z_{p}: {len(self.mstd_configs_by_prime[p]):,} configs")
        print(
            f"Certification (subject to the implemented collision, reduction, and "
            f"interchangeability assumptions): enumeration complete for primes "
            f"p > {self.global_max_prime_dividing}; for additional MSTD search, "
            f"check primes p <= {self.global_max_prime_dividing} externally."
        )
        if self.mstd_configs_by_prime:
            print("For manuscript 'unique up to affine transformation' counts, "
                  "call A.affine_classes_by_prime() (postprocessing).")
        if self.store_for_primes_larger_than is not None:
            print(f"Reduction-prime -> top-level case tracking "
                  f"(primes p >= {self.store_for_primes_larger_than}):")
            for p in sorted(self.reduction_cases_by_prime):
                print(f"  p={p}: {len(self.reduction_cases_by_prime[p])} top-level case(s)")
        print("=" * 80)
        return {
            "max_reduction": self.global_max_reduction,
            "max_prime_dividing": self.global_max_prime_dividing,
            "mstd_configs_by_prime": dict(self.mstd_configs_by_prime),
            "high_depth_mstd_configs": self.high_depth_mstd_configs,
            "mstd_integer_configs": self.mstd_integer_configs,
            "multivar_mstd_configs": self.multivar_mstd_configs,
            "reduction_cases_by_prime": {p: set(v) for p, v in self.reduction_cases_by_prime.items()},
        }

In [8]:
A = Analyzer(
    config_strings= ['0', 'a', 'b', 'c', 'd', 'e', 'd-e', 'f'], #This is the input configuration, must input each as an expression of variables inside a ''.
    status_every_seconds=120.0, #This is how often the program prints out a status report.
    checkpoint_path='/tmp/mstd_checkpoint599403.pkl', #This is the pkl file which stores the program's progress.
    checkpoint_every_seconds=5000.0,  #This is how often the program saves its progress in the checkpoint_path. A lower value will slow down the program since saving progress does not take an insignificant amount of time.
    multivar_prime_filter= 3,      #This is our beta parameter. Inputting a value p will allow the program to assume the inputted configuration is in a field of characteristic p.
    interchangeable_groups=[['a', 'b', 'c','f']]   #This is where we input the list of interchangeable variables. Note that for the program, two variables are interchangeable if swapping them does not change the configuration.
)
results = A.run()                            

Initial config: [0, a, b, c, d, d+2*e, e, f]
Variables: ['a', 'b', 'c', 'd', 'e', 'f']
Top-level collision keys (after interchangeability): 36
Max depth: 6
Checkpoint:  every 5000s to /tmp/mstd_checkpoint599403.pkl

RANDOM EXAMPLE BRANCH (for sanity-checking)
--- depth 0 ---
  config: [0, a, b, c, d, d+2*e, e, f]
  vars left: ['a', 'b', 'c', 'd', 'e', 'f'] (count=6)
  collision keys available: 36
  chose key: a+2*b+2*d
--- depth 1 ---
  config: [0, b, b+d, c, d, d+2*e, e, f]
  vars left: ['b', 'c', 'd', 'e', 'f'] (count=5)
  collision keys available: 94
  chose key: d+e+f
--- depth 2 ---
  config: [0, b, b+d, c, d, d+2*e, 2*d+2*e, e]
  vars left: ['b', 'c', 'd', 'e'] (count=4)
  collision keys available: 30
  chose key: b+c+2*d+2*e
--- depth 3 ---
  config: [0, b, b+d, 2*b+d+e, d, d+2*e, 2*d+2*e, e]
  vars left: ['b', 'd', 'e'] (count=3)
  collision keys available: 7
  chose key: b+e
--- depth 4 ---
  config: [0, d, d+2*e, d+2*e, d+2*e, 2*d+2*e, e, 2*e]
  vars left: ['d', 'e'] (count=2

In [6]:
"""
This code is used to determine which of the multivariable configurations the original program finds are affinely equivalent to one another.
"""
from itertools import permutations
from collections import defaultdict


def _vectors(cfg, p):
    """This function turns a config into its list of coefficient vectors in F_p^k,
    with variables sorted by name, and returns (k, vectors)."""
    vars_used = sorted({v for e in cfg for (v, _) in e})
    idx = {v: i for i, v in enumerate(vars_used)}
    k = len(vars_used)
    pts = []
    for e in cfg:
        vec = [0] * k
        for v, c in e:
            vec[idx[v]] = c % p
        pts.append(tuple(vec))
    return k, pts


def _matinv(cols, p):
    """This function returns the mod-p inverse of the k x k matrix whose columns
    are the given vectors, or None if that matrix is singular."""
    k = len(cols)
    # Build the augmented matrix [A | I], where A[i][j] = cols[j][i].
    A = [[cols[j][i] % p for j in range(k)] + [1 if i == j else 0 for j in range(k)]
         for i in range(k)]
    # Gauss-Jordan elimination over F_p.
    for c in range(k):
        piv = next((r for r in range(c, k) if A[r][c] % p), None)
        if piv is None:
            return None
        A[c], A[piv] = A[piv], A[c]
        inv = pow(A[c][c], p - 2, p)                 # modular inverse of the pivot
        A[c] = [(x * inv) % p for x in A[c]]
        for r in range(k):
            if r != c and A[r][c] % p:
                f = A[r][c]
                A[r] = [(A[r][j] - f * A[c][j]) % p for j in range(2 * k)]
    return [row[k:] for row in A]                    # right half is the inverse


def _mul(M, v, p):
    """This function multiplies the matrix M by the vector v mod p and returns the
    resulting vector."""
    k = len(M)
    return tuple(sum(M[i][j] * v[j] for j in range(k)) % p for i in range(k))


def _canonical(pts, k, p, affine):
    """This function returns the orbit-canonical (lex-smallest) form of a point set
    under GL(k,p) when affine is False, or Aff(k,p) when affine is True."""
    origin = tuple([0] * k)
    # Affine mode tries every point as the new origin; linear mode fixes 0.
    bases = pts if affine else [origin]
    best = None
    for base in bases:
        rel = [tuple((x[i] - base[i]) % p for i in range(k)) for x in pts]
        nz = [v for v in rel if any(v)]
        # Pick an ordered k-tuple of independent directions and map it to the
        # standard basis; the lex-smallest image over all choices is canonical.
        for frame in permutations(nz, k):
            M = _matinv(frame, p)
            if M is None:                            # frame not independent
                continue
            img = tuple(sorted(_mul(M, v, p) for v in rel))
            if best is None or img < best:
                best = img
    return best


def unique_multivar_configs(configs, p, mode="affine"):
    """This function groups configs into equivalence classes under the chosen
    symmetry ("affine" or "linear"), returning {canonical_key: [configs...]}."""
    affine = (mode == "affine")
    # Bucket by variable count first, since only same-k configs can be equivalent.
    by_k = defaultdict(list)
    parsed = {}
    for cfg in configs:
        k, pts = _vectors(cfg, p)
        by_k[k].append(cfg)
        parsed[id(cfg)] = (k, pts)
    classes = {}
    for k, cfgs in by_k.items():
        for cfg in cfgs:
            _, pts = parsed[id(cfg)]
            key = (k,) + tuple(_canonical(pts, k, p, affine))
            classes.setdefault(key, []).append(cfg)
    return classes


def format_config(cfg):
    """This function renders a config in the analyzer's tuple format as a readable
    set string, e.g. "{ 0, b, b+c, ... }"."""
    def term(e):
        if not e:
            return "0"
        return "+".join(v if c == 1 else f"{c}{v}" for v, c in e)
    return "{ " + ", ".join(term(e) for e in cfg) + " }"


def representatives(configs, p, mode="affine", pretty=False):
    """This function returns one representative config per equivalence class under
    the chosen symmetry; with pretty=True each is rendered as a readable string
    instead of the analyzer's tuple format."""
    classes = unique_multivar_configs(configs, p, mode)
    reps = [members[0] for members in classes.values()]
    if pretty:
        return [format_config(r) for r in reps]
    return reps

In [9]:
representatives(results['multivar_mstd_configs'], 3, mode="affine", pretty=True)

['{ 0, d, d+2e, d+2e+2f, d+f, e, 2e+f, f }']